In [22]:
import os
import logging

# ── Fix log directory ─────────────────────────────────────────────────────────
os.makedirs("/home/jovyan/work/persistent/orphanet/logs", exist_ok=True)
logging.getLogger("py4cytoscape").handlers = []
logging.getLogger("py4cytoscape").addHandler(logging.NullHandler())

import numpy as np
import pandas as pd
import py4cytoscape as p4c

Loading Javascript client ... 18483b6b-0efa-44eb-9f6e-dd99a14a77fd on https://jupyter-bridge.cytoscape.org


<IPython.core.display.Javascript object>

In [23]:
import py4cytoscape as p4c

print(p4c.cytoscape_version_info())
p4c.cytoscape_ping()

{'apiVersion': 'v1', 'cytoscapeVersion': '3.10.3', 'automationAPIVersion': '1.13.0', 'py4cytoscapeVersion': '1.13.0', 'jupyterBridgeVersion': '0.0.2'}
You are connected to Cytoscape!


'You are connected to Cytoscape!'

In [28]:
import os
import logging

# ── Fix log directory ─────────────────────────────────────────────────────────
os.makedirs("/home/jovyan/work/persistent/orphanet/logs", exist_ok=True)
logging.getLogger("py4cytoscape").handlers = []
logging.getLogger("py4cytoscape").addHandler(logging.NullHandler())

import numpy as np
import pandas as pd
import py4cytoscape as p4c

pd.set_option('display.max_colwidth', None)
print("Working directory:", os.getcwd())

# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv("orphanet_gene_disease.csv")
df = df.dropna(subset=["OrphaCode", "Gene Symbol"])
df["OrphaCode"]   = df["OrphaCode"].astype(str)
df["Gene Symbol"] = df["Gene Symbol"].astype(str)
print(f"Loaded: {len(df):,} rows | "
      f"{df['OrphaCode'].nunique():,} diseases | "
      f"{df['Gene Symbol'].nunique():,} genes")

# ── Build node table ──────────────────────────────────────────────────────────
disease_nodes = (
    df[["OrphaCode", "Disorder Name"]]
    .drop_duplicates(subset="OrphaCode")
    .rename(columns={"OrphaCode": "id", "Disorder Name": "label"})
    .assign(group="disease", type="disease")   # ← add type="disease"
)

gene_nodes = (
    pd.DataFrame({"id": df["Gene Symbol"].unique()})
    .assign(label=lambda x: x["id"], group="gene", type="gene")   # ← add type="gene"
)

all_nodes = pd.concat([disease_nodes, gene_nodes], ignore_index=True)
print(f"Nodes: {len(all_nodes):,} ({len(disease_nodes):,} diseases + {len(gene_nodes):,} genes)")

# ── Build edge table ──────────────────────────────────────────────────────────
all_edges = (
    df[["OrphaCode", "Gene Symbol"]]
    .drop_duplicates()
    .rename(columns={"OrphaCode": "source", "Gene Symbol": "target"})
    .assign(interaction="associated_with", weight=1)
)
print(f"Edges: {len(all_edges):,}")

# ── Connect to Cytoscape ──────────────────────────────────────────────────────
p4c.cytoscape_ping()
print("Connected to Cytoscape")

# ── Create network ────────────────────────────────────────────────────────────
suid = p4c.create_network_from_data_frames(
    nodes=all_nodes,
    edges=all_edges,
    source_id_list="source",
    target_id_list="target",
    interaction_type_list="interaction",
    node_id_list="id",
    title="Orphanet Disease-Gene Network",
    collection="AOP-RareDiseases-KG",
)
print(f"Network created — SUID: {suid}")

# ── Load node attributes ──────────────────────────────────────────────────────
p4c.load_table_data(
    data=all_nodes[["id", "label", "group", "type"]],   # ← add "type"
    data_key_column="id",
    table="node",
    table_key_column="name",
    network=suid,
)
print("Node attributes loaded")

# ── Save ──────────────────────────────────────────────────────────────────────
p4c.save_session("orphanet_disease_gene.cys")
print("Saved → orphanet_disease_gene.cys")
print(f"\nSummary — Nodes: {len(all_nodes):,} | Edges: {len(all_edges):,}")

Working directory: /home/jovyan/work/persistent/AOP-RareDiseases-KG/orphanet
Loaded: 8,164 rows | 3,961 diseases | 4,460 genes
Nodes: 8,421 (3,961 diseases + 4,460 genes)
Edges: 8,129
You are connected to Cytoscape!
Connected to Cytoscape
Applying default style...
Applying preferred layout
Network created — SUID: 729294
Node attributes loaded
Saved → orphanet_disease_gene.cys

Summary — Nodes: 8,421 | Edges: 8,129
